# BƯỚC 2: CHUẨN HÓA ĐẶC TRƯNG (FEATURE SCALING & NORMALIZATION)

## Vì sao chuẩn hóa là bắt buộc với dữ liệu khách hàng?
Ba đặc trưng phân cụm có đơn vị và thang đo khác nhau. Bảng thống kê bên dưới được tính trực tiếp từ tập Train.

Nếu không chuẩn hóa, khoảng cách Euclidean trong K-Means / Hierarchical / DBSCAN sẽ bị `Income` chi phối gần như tuyệt đối.

## Chiến lược đối chiếu & nguyên tắc chống Data Leakage
1. So sánh 4 trạng thái thang đo: `Gốc (Raw)`, `StandardScaler` ($Z = (x - \mu)/\sigma$), `MinMaxScaler` ($[0, 1]$) và `RobustScaler` (dựa trên Median & IQR).
2. **Chỉ `.fit()` trên tập Train (80%)**, sau đó `.transform()` trên Train và Holdout Test (20%).
3. Hậu kiểm tiêu chí hậu biến đổi (`Mean ≈ 0, Std ≈ 1` cho Standard; độ nhạy vượt biên `[0, 1]` trên Test của MinMax).
4. Đối chiếu thực nghiệm tác động của 3 scaler lên chất lượng K-Means.

In [ ]:
# Import các module chuẩn hóa và nạp tập Train / Holdout Test đã làm sạch từ thư mục riêng của notebook
import json
import sys
import joblib
import numpy as np
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    NOTEBOOK_FIGURES_DIR,
    NOTEBOOK_METRICS_DIR,
    NOTEBOOK_PROCESSED_DIR,
    NOTEBOOK_SAVED_MODELS_DIR,
    NUMERICAL_FEATURES,
)
from src.evaluation.visualization import plot_scaling_comparison
from src.features.scaling import (
    FeatureScaler,
    compare_scalers,
    evaluate_scalers_on_clustering,
)

NOTEBOOK_METRICS_DIR.mkdir(parents=True, exist_ok=True)
train_df = pd.read_csv(NOTEBOOK_PROCESSED_DIR / "train.csv")
test_df = pd.read_csv(NOTEBOOK_PROCESSED_DIR / "test.csv")
print(f"Đã nạp tập Train: {len(train_df):,} dòng | Tập Test: {len(test_df):,} dòng")

### 2.1 So sánh thống kê phân phối trên 4 thang đo (Raw, StandardScaler, MinMaxScaler, RobustScaler)
Bảng dưới bổ sung **Median**, **IQR** và **Skewness** để thấy rõ sự khác biệt về độ lệch và khả năng kháng ngoại lai giữa 3 bộ chuẩn hóa.

In [ ]:
scaler_comparison_df = compare_scalers(train_df)
scaler_comparison_df.to_csv(NOTEBOOK_METRICS_DIR / "scaler_comparison.csv", index=False)
scaler_comparison_df

### 2.2 Fit StandardScaler chỉ trên Train, kiểm tra tham số học được & hậu kiểm Train/Test
Đồng thời đối chiếu `MinMaxScaler` trên tập Holdout Test để quan sát hiện tượng mẫu Test vượt khỏi đoạn `[0, 1]` khi có giá trị ngoài cận Min/Max của Train.

In [ ]:
scaler = FeatureScaler(method="standard")
train_matrix = scaler.fit_transform(train_df)
test_matrix = scaler.transform(test_df)

scaler_params_df = scaler.get_learned_params()
scaler_params_df.to_csv(NOTEBOOK_METRICS_DIR / "scaler_params.csv", index=False)
print("--- Tham số StandardScaler học chỉ từ tập Train ---")
print(scaler_params_df.to_string(index=False))

minmax_reference_scaler = FeatureScaler(method="minmax").fit(train_df)
verification_df = pd.concat(
    [
        scaler.verify_scaled_matrix(train_matrix, is_train=True),
        scaler.verify_scaled_matrix(test_matrix, is_train=False),
        minmax_reference_scaler.verify_scaled_matrix(minmax_reference_scaler.transform(test_df), is_train=False),
    ],
    ignore_index=True,
)
verification_df.to_csv(NOTEBOOK_METRICS_DIR / "scaler_verification.csv", index=False)
print("--- Hậu kiểm hậu điều kiện trên Train và Holdout Test ---")
verification_df

### 2.3 Khảo sát tác động của 3 bộ chuẩn hóa lên chất lượng phân cụm K-Means
Giữ cố định tập Train/Test, 3 đặc trưng số và thuật toán `K-Means (k-means++)` để đánh giá định lượng sự đánh đổi giữa `StandardScaler`, `MinMaxScaler` và `RobustScaler`.

In [ ]:
scaler_clustering_df = evaluate_scalers_on_clustering(train_df, test_df=test_df)
scaler_clustering_df.to_csv(NOTEBOOK_METRICS_DIR / "scaler_clustering_comparison.csv", index=False)
scaler_clustering_df

### 2.4 Trực quan hóa phân phối 4 trạng thái & kiểm tra tính khả nghịch (`inverse_transform`)
`StandardScaler` là phương án chính theo Issue #1. Bảng thực nghiệm ở trên ghi kết quả riêng cho từng phương pháp.

In [ ]:
plot_scaling_comparison(train_df, train_matrix, output_dir=NOTEBOOK_FIGURES_DIR)

restored_matrix = scaler.inverse_transform(test_matrix[:3])
print("--- Kiểm tra khả năng quy đổi ngược inverse_transform (3 mẫu Test đầu) ---")
print("Giá trị gốc :\n", test_df[NUMERICAL_FEATURES].head(3).to_numpy())
print("Khôi phục lại:\n", restored_matrix.round(4))

In [ ]:
# Bàn giao ma trận đầy đủ theo đúng thứ tự ID trong df_clean
df_clean = pd.read_csv(NOTEBOOK_PROCESSED_DIR / "full_customers.csv")
X_scaled = scaler.transform(df_clean)
assert X_scaled.shape == (len(df_clean), 3)
assert np.isfinite(X_scaled).all()
assert np.allclose(
    scaler.inverse_transform(X_scaled), df_clean[NUMERICAL_FEATURES].to_numpy()
)
np.save(NOTEBOOK_PROCESSED_DIR / "X_scaled.npy", X_scaled)
df_clean[["ID"]].to_csv(NOTEBOOK_PROCESSED_DIR / "scaled_customer_ids.csv", index=False)

metadata = json.loads(
    (NOTEBOOK_PROCESSED_DIR / "preprocessing_metadata.json").read_text(encoding="utf-8")
)
metadata["feature_cols"] = list(scaler.feature_cols)
NOTEBOOK_SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {"scaler": scaler, "metadata": metadata},
    NOTEBOOK_SAVED_MODELS_DIR / "feature_scaler.joblib",
)
print(f"Ma trận bàn giao: {X_scaled.shape}, trung vị Income Train: {metadata['feature_medians']['Income']:,.2f}")